# Deep Learning Midterm Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)
## (Preprocessing Portion of Notebook)

Author: Thomas Kong

NetId: tk2558

Goal: Create train_compressed.csv by processing the train.csv data

**Important:** Make sure to upload train.csv as an input for the notebook to access! (In this notebook, train.csv is accessed throught the path: /kaggle/input/datasets/tk2558/train-prompt/train.csv)

## Referenced Data and Docs

### Dataset resources
- Provided train.csv

### Qwen 2B fine-tuning references
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

### **Section 0: Installing Necessary Packages**

> Make sure all packages and their versions are available and correct

> Uncomment whatever is needed to install for notebook environment

> Make sure train.csv and test.csv is uploaded for notebook to access

In [ ]:
# Uncomment the following in a fresh Kaggle notebook environment.
%pip install -q unsloth datasets trl transformers==4.56.2 accelerate peft bitsandbytes pandas lxml ftfy svgpathtools

# Install Node.js (if not already available)
# !apt-get update -y
# !apt-get install -y nodejs npm

# Install SVGO globally
!npm install -g svgo

In [ ]:
import unsloth, transformers, trl
# CHECK AVAILABLE AND CORRECT VERSIONS
print(transformers.__version__)
print(trl.__version__)
print(unsloth.__version__)

In [ ]:
# Install Node.js (if not already available)
!node -v # Verify Node
!svgo --version # Verify SVGO installation

### **Section 1: Configuration**

> Initialize variables for the notebook and models

> After running all cellblocks in Section 1, you can skip to 2B if you are already using pre-installed training_compressed.csv or skip to Section 7 if you are using pretrained fine-tuned model provided.

In [ ]:
import os
import re
import time
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset, Dataset
import hashlib, random, numpy as np, torch

NETID = "tk2558"
SEED  = int(hashlib.sha256(NETID.encode()).hexdigest(), 16) % 10000
print(f"NetID: {NETID}  |  Seed: {SEED}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Core training config.
# "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit" (Current)

CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",  # Verify exact ID from the linked Unsloth notebook.
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 64,
    "learning_rate": 2e-4, #2e-4,
    "num_train_epochs": 1, #1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.05,
    "warmup_steps": 200, #10,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 30000, # Adjust as necessary for GPU 
    "eval_size": 0.02,
    "output_dir": "/kaggle/working/qwen2b_svg_lora",
}

CONFIG

In [ ]:
import pandas as pd

# LOAD DATASET
df = pd.read_csv("/kaggle/input/datasets/tk2558/train-prompt/train.csv", engine="python", on_bad_lines='skip')
assert all(col in df.columns for col in ["id", "prompt", "svg"])

print("Total samples:", len(df))
df = df.dropna(subset=["prompt", "svg"])
df.head()

### **Section 2A: Preprocessing Data Part 1**

> (Can skip this part if you've already installed train_compression.csv and notebook has access to it)

> This part is necessary for creating train_compression.csv, a more optimized and streamline version of the train.csv for model to train. This is the Data Compression Pipeline

> In this Data Compression Pipeline we: compress SVG to use less tokens and scale every SVG to fill a 256x256 Canvas


In [ ]:
with open("/kaggle/working/svgo.config.transform.js", "w") as f:
    f.write("""
export default {
  multipass: true,
  floatPrecision: 4,
  plugins: [
    {
      name: "preset-default",
      params: {
        overrides: {
          removeViewBox: false,
          removeDimensions: false,
          mergePaths: false,
          convertShapeToPath: false,
          cleanupNumericValues: false
        }
      }
    },
    {
      name: "convertTransform"
    },
    {
      name: "sortAttrs"
    }
  ]
};
""")

print("svgo.config.transform.js FILE CREATED")

In [ ]:
with open("/kaggle/working/svgo.config.compress.js", "w") as f:
    f.write("""
export default {
  multipass: true,
  floatPrecision: 2,
  plugins: [
    {
      name: "preset-default",
      params: {
        overrides: {
          removeViewBox: false,
          removeDimensions: false,
          mergePaths: false,
          convertShapeToPath: false
        }
      }
    },

    {
      name: "cleanupNumericValues",
      params: { floatPrecision: 2 }
    },

    {
      name: "sortAttrs"
    }
  ]
};
""")

print("svgo.config.compress.JS FILE CREATED")

In [ ]:
import os

BASE_DIR = "/kaggle/working/svg_pipeline"

IN_DIR = f"{BASE_DIR}/svg_in"
MID_DIR = f"{BASE_DIR}/svg_mid"
OUT_DIR = f"{BASE_DIR}/svg_out"
FINAL_DIR = f"{BASE_DIR}/svg_final"

os.makedirs(IN_DIR, exist_ok=True)
os.makedirs(MID_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

print("FOLDER DIRECTORY CREATED")

In [ ]:
import re
import subprocess
import tempfile
import os
from transformers import AutoTokenizer

# SVG cleanup
def basic_svg_cleanup(svg):
    if not isinstance(svg, str):
        return ""

    svg = re.sub(r"<\?xml.*?\?>", "", svg) # Remove XML header if exists
    svg = re.sub(r"<!--.*?-->", "", svg, flags=re.DOTALL) # Remove comments
    svg = re.sub(r"<metadata.*?</metadata>", "", svg, flags=re.DOTALL) # Remove metadata blocks
    svg = re.sub(r"\s+", " ", svg) # Normalize whitespace
    svg = re.sub(r">\s+<", "><", svg) # Remove spaces between tags
    svg = re.sub(r"\s*=\s*", "=", svg) # Remove unnecessary quotes spacing

    return svg.strip()

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
def count_svg_tokens(svg):
    return len(tokenizer(svg)["input_ids"])

In [ ]:
import re
from svgpathtools import parse_path, Path
from lxml import etree

def _parse_translate(transform_str):
    # Extract (tx, ty) from a translate(...) transform string. Returns (0,0) if not found
    m = re.search(r'translate\(([^)]+)\)', transform_str or '')
    if not m:
        return 0.0, 0.0
    nums = re.findall(r'-?[\d.]+', m.group(1))
    tx = float(nums[0]) if len(nums) > 0 else 0.0
    ty = float(nums[1]) if len(nums) > 1 else 0.0
    return tx, ty


def bake_path_translate(svg_string):
    try:
        root = etree.fromstring(svg_string.encode())
    except etree.XMLSyntaxError:
        return _bake_path_translate_regex(svg_string)

    def _walk(el, acc_tx, acc_ty):
        tag = etree.QName(el.tag).localname if '{' in el.tag else el.tag
        transform = el.get('transform', '')

        # Accumulate translate from this element's own transform
        own_tx, own_ty = _parse_translate(transform) if 'translate' in transform else (0.0, 0.0)
        total_tx = acc_tx + own_tx
        total_ty = acc_ty + own_ty

        if tag == 'path':
            d = el.get('d', '')
            if d and (total_tx != 0.0 or total_ty != 0.0):
                try:
                    baked = parse_path(d).translated(complex(total_tx, total_ty))
                    el.set('d', baked.d())
                except Exception:
                    pass

            # Remove the translate part from the path's OWN transform (if any).
            if own_tx != 0.0 or own_ty != 0.0:
                remaining = re.sub(r'\s*translate\([^)]+\)', '', transform).strip()
                if remaining:
                    el.set('transform', remaining)
                else:
                    del el.attrib['transform']

            # Paths are leaves for translation purposes; reset accumulator for any children
            for child in el:
                _walk(child, 0.0, 0.0)

        elif tag == 'g':
            # Process children first so they absorb the full accumulated translation
            for child in el:
                _walk(child, total_tx, total_ty)
            if own_tx != 0.0 or own_ty != 0.0:
                remaining = re.sub(r'\s*translate\([^)]+\)', '', transform).strip()
                if remaining:
                    el.set('transform', remaining)
                elif 'transform' in el.attrib:
                    del el.attrib['transform']

        else:
            # For all other elements: accumulate but do not remove the transform.
            for child in el:
                _walk(child, total_tx, total_ty)

    _walk(root, 0.0, 0.0)
    return etree.tostring(root, encoding='unicode')


def _bake_path_translate_regex(svg_string):
    # Fallback for malformed SVGs lxml cannot parse — handles only standalone <path transform=translate>
    def bake(m):
        full_tag = m.group(0)
        t_match = re.search(r'transform="translate\(([^)]+)\)"', full_tag)
        d_match = re.search(r'(?<![a-z])d="([^"]+)"', full_tag)
        if not t_match or not d_match:
            return full_tag
        nums = re.findall(r'-?[\d.]+', t_match.group(1))
        tx = float(nums[0]) if len(nums) > 0 else 0.0
        ty = float(nums[1]) if len(nums) > 1 else 0.0
        try:
            path = parse_path(d_match.group(1))
            baked = path.translated(complex(tx, ty))
            new_tag = re.sub(r'(?<![a-z])d="[^"]+"', f'd="{baked.d()}"', full_tag)
            new_tag = re.sub(r'\s*transform="translate\([^)]+\)"', '', new_tag)
            return new_tag
        except Exception:
            return full_tag
    return re.sub(r'<path\b[^>]*(?:/>|>)', bake, svg_string)


def normalize_paths(svg_string):
    if re.search(r'<g[^>]+transform=[^>]*translate', svg_string):
        return svg_string

    paths = re.findall(r'(?<![a-z])d="([^"]+)"', svg_string)
    if not paths:
        return svg_string

    xmin, ymin = float('inf'), float('inf')
    parsed_paths = []
    valid_old_ds = []

    for d in paths:
        try:
            p = parse_path(d)
        except Exception:
            continue
        bbox = p.bbox()
        if bbox[0] == bbox[1] and bbox[2] == bbox[3]:
            svg_string = re.sub(
                r'(?<![a-z])d="' + re.escape(d) + r'"',
                'd=""', svg_string, count=1,
            )
            continue
        xmin = min(xmin, bbox[0])
        ymin = min(ymin, bbox[2])
        parsed_paths.append(p)
        valid_old_ds.append(d)

    if xmin == float('inf'):
        return svg_string

    # Only shift if geometry actually starts off-canvas
    if xmin >= 0 and ymin >= 0:
        return svg_string

    for old_d, p in zip(valid_old_ds, parsed_paths):
        new_p = p.translated(complex(-xmin, -ymin))
        svg_string = re.sub(
            r'(?<![a-z])d="' + re.escape(old_d) + r'"',
            f'd="{new_p.d()}"', svg_string, count=1,
        )

    return svg_string

In [ ]:
import re
from svgpathtools import parse_path, Path

def scale_svg_to_256(svg_string):
    """
    Scale an SVG so its largest dimension fills 256 px, updating all geometry.
    Call normalize_paths() BEFORE this function, not inside it.
    """

    # -- 1. Parse canvas size -------------------------------------------- #
    viewbox_match = re.search(
        r'viewBox="(-?[\d.]+)\s+(-?[\d.]+)\s+([\d.]+)\s+([\d.]+)"',
        svg_string,
    )
    if viewbox_match:
        orig_w = float(viewbox_match.group(3))
        orig_h = float(viewbox_match.group(4))
    else:
        w_match = re.search(r'width="([\d.]+)"', svg_string)
        h_match = re.search(r'height="([\d.]+)"', svg_string)
        orig_w = float(w_match.group(1)) if w_match else 256.0
        orig_h = float(h_match.group(1)) if h_match else 256.0

    scale = 256.0 / max(orig_w, orig_h)
    new_w  = round(orig_w * scale, 4)
    new_h  = round(orig_h * scale, 4)

    def fmt(v):
        return int(v) if v == int(v) else round(v, 4)

    # -- 2. Scale transform coordinates ---------------------------------- #
    def scale_translate(m):
        nums = re.findall(r"-?[\d.]+", m.group(1))
        tx = round(float(nums[0]) * scale, 4) if nums else 0
        ty = round(float(nums[1]) * scale, 4) if len(nums) > 1 else 0
        return f"translate({tx} {ty})"

    def scale_rotate(m):
        nums = re.findall(r"-?[\d.]+", m.group(1))
        angle = nums[0]                          # angle: never scaled
        if len(nums) == 3:
            cx = round(float(nums[1]) * scale, 4)
            cy = round(float(nums[2]) * scale, 4)
            return f"rotate({angle} {cx} {cy})"
        return f"rotate({angle})"

    def scale_matrix(m):
        nums = re.findall(r"-?[\d.]+", m.group(1))
        if len(nums) == 6:
            a, b, c, d, e, f = nums
            e2 = round(float(e) * scale, 4)      # e: x-translation
            f2 = round(float(f) * scale, 4)      # f: y-translation
            return f"matrix({a} {b} {c} {d} {e2} {f2})"
        return m.group(0)

    svg_string = re.sub(r"translate\(([^)]+)\)", scale_translate, svg_string)
    svg_string = re.sub(r"rotate\(([^)]+)\)",    scale_rotate,    svg_string)
    svg_string = re.sub(r"matrix\(([^)]+)\)",    scale_matrix,    svg_string)

    # -- 3. Scale path d= (command-aware, arc flags preserved) ----------- #
    # (?<![a-z]) prevents matching id=, method=, href= as path data
    def scale_path_d(m):
        return f'd="{_scale_path_commands(m.group(1), scale)}"'
    svg_string = re.sub(r'(?<![a-z])d="([^"]+)"', scale_path_d, svg_string)

    # -- 4. Scale polyline/polygon points= ------------------------------- #
    def scale_points(m):
        def sn(n): return str(round(float(n.group()) * scale, 4))
        return f'points="{re.sub(r"-?[\d.]+", sn, m.group(1))}"'
    svg_string = re.sub(r'points="([^"]+)"', scale_points, svg_string)

    # -- 5. Scale presentation attributes -------------------------------- #
    def make_scaler(attr):
        def r(m): return f'{m.group(1)}="{fmt(float(m.group(2)) * scale)}"'
        return r

    # Compound attrs first — prevents (x)= from double-scaling cx/dx/rx etc.
    for attr in ["cx", "cy", "rx", "ry", "dx", "dy", "x1", "y1", "x2", "y2"]:
        svg_string = re.sub(fr'({attr})="(-?[\d.]+)"', make_scaler(attr), svg_string)

    svg_string = re.sub(r'(?<![a-z])(x)="(-?[\d.]+)"',          make_scaler("x"), svg_string)
    svg_string = re.sub(r'(?<![a-z])(y)="(-?[\d.]+)"',          make_scaler("y"), svg_string)
    svg_string = re.sub(r'(?<![a-z])(r)(?![a-z])="(-?[\d.]+)"', make_scaler("r"), svg_string)

    for attr in ["width", "height"]:  # stroke-width caught here too
        svg_string = re.sub(fr'({attr})="(-?[\d.]+)"', make_scaler(attr), svg_string)

    svg_string = re.sub(r'(font-size)="(-?[\d.]+)"', make_scaler("font-size"), svg_string)

    # -- 6. Always update viewBox ---------------------------------------- #
    new_vb = f"0 0 {fmt(new_w)} {fmt(new_h)}"
    if viewbox_match:
        svg_string = re.sub(
            r'(<svg[^>]*\s)viewBox="[^"]*"',
            lambda m: f'{m.group(1)}viewBox="{new_vb}"',
            svg_string, count=1,
        )

    # -- 7. Force root width/height -------------------------------------- #
    svg_string = re.sub(
        r'(<svg[^>]*\s)width="[^"]*"',
        lambda m: f'{m.group(1)}width="{fmt(new_w)}"',
        svg_string, count=1,
    )
    svg_string = re.sub(
        r'(<svg[^>]*\s)height="[^"]*"',
        lambda m: f'{m.group(1)}height="{fmt(new_h)}"',
        svg_string, count=1,
    )

    # Clean up empty paths left by normalize_paths
    svg_string = re.sub(r'<path[^>]*d=""[^>]*/>', '', svg_string)
    svg_string = re.sub(r'<path[^>]*d="\s*"[^>]*></path>', '', svg_string)

    return svg_string


def _scale_path_commands(d: str, scale: float) -> str:
    # Command-aware path scaler. Arc flags are never multiplied by scale.
    tokens = re.findall(
        r"[MmZzLlHhVvCcSsQqTtAa]"
        r"|[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?",
        d,
    )
    out = []
    i   = 0

    def s(v): return str(round(float(v) * scale, 4))

    while i < len(tokens):
        tok = tokens[i]
        if tok in "Zz":
            out.append(tok); i += 1
        elif tok in "MmLlTt":
            out.append(tok); i += 1
            out.append(s(tokens[i])); i += 1
            out.append(s(tokens[i])); i += 1
        elif tok in "Hh":
            out.append(tok); i += 1
            out.append(s(tokens[i])); i += 1
        elif tok in "Vv":
            out.append(tok); i += 1
            out.append(s(tokens[i])); i += 1
        elif tok in "Cc":
            out.append(tok); i += 1
            for _ in range(6): out.append(s(tokens[i])); i += 1
        elif tok in "SsQq":
            out.append(tok); i += 1
            for _ in range(4): out.append(s(tokens[i])); i += 1
        elif tok in "Aa":
            out.append(tok); i += 1
            out.append(s(tokens[i])); i += 1   # rx       -- scale
            out.append(s(tokens[i])); i += 1   # ry       -- scale
            out.append(tokens[i]);    i += 1   # rotation -- NO scale
            out.append(tokens[i]);    i += 1   # large-arc -- NO scale
            out.append(tokens[i]);    i += 1   # sweep    -- NO scale
            out.append(s(tokens[i])); i += 1   # x        -- scale
            out.append(s(tokens[i])); i += 1   # y        -- scale
        else:
            out.append(s(tok)); i += 1         # implicit repeat

    return " ".join(out)

In [ ]:
from svgpathtools import parse_path, Path

def clean_degenerate_segments(svg_string):
    # Remove zero-length segments from all paths
    def clean_path(match):
        d = match.group(1)
        try:
            path = parse_path(d)
        except Exception:
            return match.group(0)
        new_segs = [seg for seg in path if abs(seg.start - seg.end) >= 1e-6]
        if not new_segs:
            return match.group(0)
        return f'd="{Path(*new_segs).d()}"'
    # (?<![a-z]) guard prevents matching id=, href= etc.
    return re.sub(r'(?<![a-z])d="([^"]+)"', clean_path, svg_string)


def round_path_numbers(svg_string, precision=2):
    """Round all floats in the SVG to reduce token count."""
    def round_num(m):
        return str(round(float(m.group()), precision))
    return re.sub(r'(?<![a-zA-Z])-?\d+\.\d+', round_num, svg_string)


def remove_redundant_transforms(svg_string):
    def remove(m):
        nums = re.findall(r'-?[\d.]+', m.group(1))
        if all(abs(float(n)) < 1e-6 for n in nums):
            return ''       # zero translate -> drop it
        return m.group(0)  # non-zero -> keep it

    return re.sub(r'\s*transform="(translate\([^)]+\))"', remove, svg_string)


In [ ]:
# COMPRESSION PIPELINE A (BASIC CLEAN UP)
from tqdm import tqdm

valid_indices = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    svg = basic_svg_cleanup(row["svg"]) # BASIC CLEANUP
    if not svg.startswith("<svg"):
        continue

    file_path = f"{IN_DIR}/{i}.svg"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(svg)

    valid_indices.append(i)

print("Saved SVGs:", len(valid_indices))

In [ ]:
%%capture
# Just to ignore Streaming Output
!svgo -f /kaggle/working/svg_pipeline/svg_in \
      -o /kaggle/working/svg_pipeline/svg_mid \
      --multipass \
      --config=/kaggle/working/svgo.config.transform.js

In [ ]:
optimized_map = {}

for i in valid_indices:
    path = f"{OUT_DIR}/{i}.svg"

    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            svg = f.read()

            # Basic validation
            if svg.startswith("<svg") and svg.endswith("</svg>"):
                optimized_map[i] = svg

In [ ]:
# COMPRESSION PIPELINE B (SCALE)

scaled_indices = []
failed_indices = []

for i in valid_indices:
    path = f"{MID_DIR}/{i}.svg"
    if not os.path.exists(path):
        continue
    with open(path, "r", encoding="utf-8") as f:
        svg = f.read()

    try:
        svg = bake_path_translate(svg)              # 1. absorb any translate() SVGO missed
        svg = normalize_paths(svg)                  # 2. shift to (0,0) BEFORE scaling
        svg = scale_svg_to_256(svg)                 # 3. scale to 256 canvas
        svg = clean_degenerate_segments(svg)        # 4. remove zero-length segments
        svg = remove_redundant_transforms(svg)      # 5. drop zero-value transforms
        svg = round_path_numbers(svg, precision=2)  # 6. round for token savings

    except Exception as e:
        failed_indices.append((i, str(e)))
        continue

    out_path = f"{OUT_DIR}/{i}.svg"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(svg)

    scaled_indices.append(i)

print(f"Scaled:  {len(scaled_indices)}")
print(f"Failed:  {len(failed_indices)}")

In [ ]:
%%capture
# Just to ignore Streaming Output
!svgo -f /kaggle/working/svg_pipeline/svg_out \
      -o /kaggle/working/svg_pipeline/svg_final \
      --multipass \
      --config=/kaggle/working/svgo.config.compress.js

In [ ]:
optimized_map = {}

for i in valid_indices:
    path = f"{FINAL_DIR}/{i}.svg"

    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            svg = f.read()

            # Basic validation
            if svg.startswith("<svg") and svg.endswith("</svg>"):
                optimized_map[i] = svg

In [ ]:
MAX_SVG_TOKENS = 1024 #512
compressed_rows = []

for i, row in df.iterrows():
    if i not in optimized_map:
        continue

    svg = optimized_map[i]
    token_len = count_svg_tokens(svg)

    if token_len <= MAX_SVG_TOKENS:
        compressed_rows.append({
            "id": row["id"],
            "prompt": row["prompt"],
            "svg": svg,
            "tokens": token_len
        })

In [ ]:
compressed_df = pd.DataFrame(compressed_rows)

print("Original:", len(df))
print("After compression:", len(compressed_df))
print("\nToken stats:")
print(compressed_df["tokens"].describe())

In [ ]:
compressed_df.to_csv("/kaggle/working/train_compressed.csv", index=False)